# 01 · Baseline: TF-IDF + Logistic Regression

Fixed stratified validation split (saved to `splits/` and reused by later models),
TF-IDF word / char n-grams + LogisticRegression, small parameter search on macro-F1,
per-class metrics, confusion matrix, `submission_baseline.csv`.
Shared code (data loading, split, submission checks) lives in `src/common.py`.

Vectorizers are fitted **on the train part only** (no leakage from validation or test).

## Setup: get the code (Kaggle clones the repo, locally uses `../src`)

In [ ]:
import os
import subprocess
import sys

REPO = "github.com/mohinska/ai_hw3_AIGenTextDetection.git"
BRANCH = "main"

if os.path.exists("/kaggle/input"):
    repo_dir = "/tmp/repo"
    token = ""
    try:  # needed only if the repo is private: Kaggle Secret GITHUB_TOKEN
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN") + "@"
    except Exception:
        pass
    subprocess.run(["rm", "-rf", repo_dir], check=True)
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH, f"https://{token}{REPO}", repo_dir], check=True)
    sys.path.insert(0, os.path.join(repo_dir, "src"))
else:
    sys.path.insert(0, os.path.abspath("../src" if os.path.exists("../src") else "src"))

In [ ]:
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, f1_score

import common
from common import LABELS, TEXT_COL, LABEL_COL, SEED

warnings.filterwarnings("ignore", category=UserWarning)

## Config

In [ ]:
N_TRAIN = 60_000         # stratified subsample of the train part used for fitting (None = all)
MAX_CHARS = 5_000        # truncate very long texts to keep char n-grams tractable
REFIT_ON_TRAIN_PLUS_VAL = False  # True: refit best config on train+val before predicting test

OUT_DIR = os.path.join(common.WORK_DIR, "outputs")
SUBMISSION_PATH = os.path.join(common.WORK_DIR, "submission_baseline.csv")
os.makedirs(OUT_DIR, exist_ok=True)

## Load data

In [ ]:
train_df, test_df, sample_sub = common.load_data()
LANG_COL = common.lang_col(train_df)

assert set(train_df[LABEL_COL].unique()) == set(LABELS), train_df[LABEL_COL].unique()
print("train:", train_df.shape, "| test:", test_df.shape, "| columns:", list(train_df.columns))
train_df[LABEL_COL].value_counts(normalize=True).round(3)

## Fixed validation split

`common.split_train_val`: 10% stratified by label (and language), seed 42. It is deterministic,
so every later notebook gets exactly the same validation rows. Ids are also saved to `outputs/`.

In [ ]:
trn, val = common.split_train_val(train_df)
val[["id"]].to_csv(os.path.join(OUT_DIR, f"val_ids_seed{SEED}.csv"), index=False)
trn = common.subsample(trn, N_TRAIN)
print(f"train part: {len(trn):,} | val: {len(val):,}")

In [ ]:
def prep(texts):
    return texts.fillna("").astype(str).str.slice(0, MAX_CHARS)

X_trn_txt, y_trn = prep(trn[TEXT_COL]), trn[LABEL_COL].values
X_val_txt, y_val = prep(val[TEXT_COL]), val[LABEL_COL].values
X_test_txt = prep(test_df[TEXT_COL])

## Parameter search

Each vectorizer is fitted once on the train part; the validation set is only transformed.
For each feature set a small grid over the regularisation strength `C` is evaluated by macro-F1.

In [ ]:
VECTORIZERS = {
    "word_1-2": dict(analyzer="word", ngram_range=(1, 2), min_df=3, max_features=300_000,
                     sublinear_tf=True, lowercase=True),
    "char_wb_2-5": dict(analyzer="char_wb", ngram_range=(2, 5), min_df=5, max_features=300_000,
                        sublinear_tf=True, lowercase=False),
}
C_GRID = [0.5, 2.0, 8.0]


def fit_vectorizers(names, train_texts):
    vecs = {n: TfidfVectorizer(dtype=np.float32, **VECTORIZERS[n]).fit(train_texts) for n in names}
    return vecs


def transform(vecs, texts):
    return hstack([v.transform(texts) for v in vecs.values()]).tocsr()


def make_clf(C):
    return LogisticRegression(C=C, max_iter=2000, random_state=SEED)


feature_sets = {"word_1-2": ["word_1-2"],
                "char_wb_2-5": ["char_wb_2-5"],
                "word+char": ["word_1-2", "char_wb_2-5"]}

fitted = fit_vectorizers(VECTORIZERS.keys(), X_trn_txt)   # fit on train part only
results = []
for fs_name, vec_names in feature_sets.items():
    vecs = {n: fitted[n] for n in vec_names}
    Xtr, Xva = transform(vecs, X_trn_txt), transform(vecs, X_val_txt)
    for C in C_GRID:
        t0 = time.time()
        clf = make_clf(C).fit(Xtr, y_trn)
        f1 = f1_score(y_val, clf.predict(Xva), average="macro")
        results.append({"features": fs_name, "C": C, "n_feats": Xtr.shape[1],
                        "macro_f1": round(f1, 4), "fit_s": round(time.time() - t0, 1)})

results_df = pd.DataFrame(results).sort_values("macro_f1", ascending=False).reset_index(drop=True)
results_df.to_csv(os.path.join(OUT_DIR, "baseline_grid.csv"), index=False)
results_df

## Best model on the validation split

In [ ]:
best = results_df.iloc[0]
best_vecs = {n: fitted[n] for n in feature_sets[best["features"]]}
Xtr, Xva = transform(best_vecs, X_trn_txt), transform(best_vecs, X_val_txt)
best_clf = make_clf(best["C"]).fit(Xtr, y_trn)

val_proba = best_clf.predict_proba(Xva)
val_pred = best_clf.classes_[val_proba.argmax(1)]
print(f"best: {best['features']}, C={best['C']} -> val macro-F1 = {f1_score(y_val, val_pred, average='macro'):.4f}\n")
print(classification_report(y_val, val_pred, labels=LABELS, digits=4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, norm, title in zip(axes, [None, "true"], ["counts", "row-normalised"]):
    ConfusionMatrixDisplay.from_predictions(y_val, val_pred, labels=LABELS, normalize=norm,
                                            values_format=".2f" if norm else "d",
                                            cmap="Blues", colorbar=False, ax=ax)
    ax.set_title(f"Baseline confusion matrix ({title})")
plt.tight_layout()
plt.show()

In [ ]:
# macro-F1 per language (if available): shows whether one language is much harder
if LANG_COL:
    per_lang = (pd.DataFrame({"lang": val[LANG_COL], "y": y_val, "p": val_pred})
                .groupby("lang").apply(lambda g: pd.Series({"n": len(g),
                                                             "macro_f1": f1_score(g.y, g.p, average="macro")}))
                .round(4))
    display(per_lang)

In [ ]:
# Save validation probabilities: reused later for ensembling / threshold tuning
val_out = pd.DataFrame(val_proba, columns=[f"p_{c}" for c in best_clf.classes_])
val_out.insert(0, "id", val["id"].values)
val_out.to_csv(os.path.join(OUT_DIR, "val_proba_baseline.csv"), index=False)

## Predict test and write submission

In [ ]:
if REFIT_ON_TRAIN_PLUS_VAL:
    full_txt = pd.concat([X_trn_txt, X_val_txt], ignore_index=True)
    full_y = np.concatenate([y_trn, y_val])
    final_vecs = fit_vectorizers(feature_sets[best["features"]], full_txt)
    final_clf = make_clf(best["C"]).fit(transform(final_vecs, full_txt), full_y)
else:
    final_vecs, final_clf = best_vecs, best_clf

test_proba = final_clf.predict_proba(transform(final_vecs, X_test_txt))
test_pred = final_clf.classes_[test_proba.argmax(1)]

test_out = pd.DataFrame(test_proba, columns=[f"p_{c}" for c in final_clf.classes_])
test_out.insert(0, "id", test_df["id"].values)
test_out.to_csv(os.path.join(OUT_DIR, "test_proba_baseline.csv"), index=False)

submission = common.save_submission(test_df["id"], test_pred, sample_sub, SUBMISSION_PATH)
print("saved", SUBMISSION_PATH, submission.shape)
submission["label"].value_counts(normalize=True).round(3)